# Decisions inside the model {#sec-decisions}

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/waelrash1/loopbricks/blob/main/notebooks/c05-decisions.ipynb)

A waiting list does not run itself. Every month someone decides how many clinics to run, whether to hire, and how hard to push the staff already in post. Those choices are the flows that move the list, so a model that leaves them out explains nothing about the list. This chapter is about writing those choices as equations: **decision rules** that use only what the decision maker can see, that bend sensibly when pressure rises, and that never ask a stock for more than it holds.

The running case gains its first people. Clinicians treat patients from the list; a manager sets a treatment target and hires towards it; and clinical productivity rises under pressure through a nonlinear effect, up to a limit. The diagram in section 1 shows the structure you will have built by the end.

- **Who it is for:** anyone who has built stocks and loops (chapters 3 and 4) and now has to model people making choices.
- **Who can skip:** experienced modellers who already write protected outflows and lookup effects by reflex can read the family tables and go straight to section 9 and the running case.
- **Prerequisites:** chapter 1 (stocks and flows), chapter 3 (the Close Gap molecule) and chapter 4 (the Smooth molecule for perception).
- **Time:** about three hours, plus the mini-project.

The chapter introduces 25 molecules. They come in seven families, each with one worked example and a compact table in which every molecule is run once.

In [ ]:
# Run this cell first. In Colab it installs the loopbricks package, which takes about ten seconds.
try:
    import loopbricks as lb
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "loopbricks @ git+https://github.com/waelrash1/loopbricks.git@main#subdirectory=python"])
    import loopbricks as lb
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Ellipse
from cycler import cycler

SEED = 5
rng = np.random.default_rng(SEED)

DATA = "data/waiting_list.csv" if os.path.exists("data/waiting_list.csv") else \
    "https://raw.githubusercontent.com/waelrash1/loopbricks/main/notebooks/data/waiting_list.csv"
wl = pd.read_csv(DATA, parse_dates=["month"])

# Okabe-Ito colours and distinct line styles: figures read in greyscale
OKABE_ITO = ["#000000", "#E69F00", "#56B4E9", "#009E73",
             "#D55E00", "#0072B2", "#CC79A7", "#F0E442"]
STYLES = ["-", "--", ":", "-.", (0, (5, 1)), (0, (3, 1, 1, 1)),
          (0, (1, 1)), (0, (5, 3))]
plt.rcParams["axes.prop_cycle"] = (cycler(color=OKABE_ITO)
                                   + cycler(linestyle=STYLES))
plt.rcParams.update({"figure.figsize": (8, 4), "axes.grid": True,
                     "grid.alpha": 0.3, "lines.linewidth": 2})
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 200)
print("loopbricks", lb.__version__)

## Learning outcomes

By the end of this chapter you will be able to:

1. **Formulate** a decision rule from the information a decision maker actually has, and separate the desired rate from the achieved one (Try it in sections 1 and 2).
2. **Build** a nonlinear effect with `TABLE` and **justify** its shape, its normal point and its limits (Try it in section 3).
3. **Diagnose** drift in an anchoring and adjustment formulation, and fix it with a protected anchor (Try it in section 4).
4. **Predict** how a split rule shares a scarce resource between competing claims (Try it in section 5).
5. **Protect** outflows so stocks cannot go negative, and **test** them at extreme conditions (Try it in section 6).
6. **Convert** between resources, action, ability and money in both directions (Try it in section 7).
7. **Explain** why productivity measured from output can mislead a staffing decision, and **assemble** the staffing model of the running case (Try it in sections 8 and 9, and the mini-project).

## 1. Decision rules from what decision makers know

A **decision rule** is the equation behind a flow that some person controls. It maps the information available at the moment of choice to an action. Three habits make such rules credible.

- **Use only information the decision maker has.** A waiting-list manager sees the list, recent referrals and the staff in post. They do not see next month's referrals or the true productivity of each clinician. In the model, the rule must read perceived or smoothed quantities, never the future and never hidden ones.
- **Assume bounded rationality, not omniscience.** People cannot solve the whole system in their heads. They use rules of thumb: start from something familiar (an **anchor**) and adjust it for the evidence, usually by too little [@tversky1974]. System dynamics writes these rules down as they are, rather than replacing them with an optimum nobody computes [@morecroft1983; @sterman1989].
- **Keep the rule robust.** Whatever the inputs, the rule must give a rate that can exist: no negative treatments, no hiring that drives staff below zero, no effect that grows without bound.

The running case so far is the waiting list from chapter 1: one stock, two flows. The data are synthetic, shaped like published Irish waiting-list counts (see chapter 10). Only the stock is observed, so referrals and treatments have to be inferred. The function below builds the case model from parts, so each section can add its own decision rules. A later part replaces an element of the same name, which is how `Capacity` will change from a constant into a rule.

In [ ]:
def S(name, init, units):
    return dict(kind="stock", name=name, init=init, units=units)


def F(name, eq, units, src=None, dst=None):
    return dict(kind="flow", name=name, eq=eq, units=units, src=src, dst=dst)


def V(name, eq, units):
    return dict(kind="var", name=name, eq=eq, units=units)


def build(*parts, name="Waiting list case", T=120, dt=0.125):
    """A case model from lists of S, F and V; later parts replace earlier
    elements that have the same name."""
    els = {}
    for part in parts:
        for e in part:
            els[e["name"]] = e
    key = {n: n.lower().replace(" ", "_") for n in els}
    stocks, flows, vars_ = [], [], []
    for n, e in els.items():
        if e["kind"] == "stock":
            stocks.append({"id": key[n], "name": n, "init": e["init"],
                           "units": e["units"], "nonNegative": True})
        elif e["kind"] == "flow":
            f = {"id": key[n], "name": n, "eq": e["eq"], "units": e["units"]}
            if e["src"]:
                f["from"] = key[e["src"]]
            if e["dst"]:
                f["to"] = key[e["dst"]]
            flows.append(f)
        else:
            vars_.append({"id": key[n], "name": n, "eq": e["eq"],
                          "units": e["units"]})
    return lb.Model.from_spec({"name": name, "timeUnit": "month", "T": T,
                               "dt": dt, "stocks": stocks, "flows": flows,
                               "vars": vars_})


BASE = [
    S("Waiting list", "32000", "patients"),
    F("Referrals", "Base_referrals + STEP(Referral_increase, 24)",
      "patients/month", dst="Waiting list"),
    F("Treatments", "MIN(Capacity, Waiting_list / Minimum_treatment_time)",
      "patients/month", src="Waiting list"),
    V("Base referrals", "4000", "patients/month"),
    V("Referral increase", "0", "patients/month"),
    V("Capacity", "3950", "patients/month"),
    V("Minimum treatment time", "0.5", "month"),
]
base = build(BASE)
print(base.equations())
b = base.run()
print(f"\nWaiting list at month 120: {b['Waiting list'].iloc[-1]:,.0f} "
      f"patients; data in December 2023: {wl['total'].iloc[-1]:,} patients")

Time is in months from January 2014, so month 120 is the end of 2023. With referrals of 4,000 patients/month against a capacity of 3,950 patients/month, the list grows by 50 patients/month and reaches 38,000 patients. The data end near 70,000 patients. A fixed capacity cannot be the whole story: someone decides capacity, and the next sections write that decision down.

The diagram shows where the chapter is going. Boxes are stocks, double arrows with a valve are flows, thin arrows carry information into the decision rules.

In [ ]:
def draw_case(ax):
    """Stock-and-flow diagram of the chapter's case model."""
    def stock(x, y, label):
        ax.add_patch(FancyBboxPatch((x - 1.1, y - 0.45), 2.2, 0.9,
                                    boxstyle="square,pad=0", fc="#E8E8E8",
                                    ec="black", lw=1.5))
        ax.text(x, y, label, ha="center", va="center", fontsize=10,
                fontweight="bold")

    def cloud(x, y):
        ax.add_patch(Ellipse((x, y), 0.9, 0.6, fc="white", ec="grey",
                             ls="--"))

    def flow(x0, x1, y, label, xv):
        ax.annotate("", xy=(x1, y), xytext=(x0, y),
                    arrowprops=dict(arrowstyle="-|>,head_width=0.4",
                                    lw=4, color="black"))
        ax.add_patch(Ellipse((xv, y), 0.35, 0.35, fc="white", ec="black",
                             lw=1.5, zorder=3))
        ax.text(xv, y - 0.45, label, ha="center", va="top", fontsize=9)

    def var(x, y, label):
        ax.text(x, y, label, ha="center", va="center", fontsize=9,
                style="italic")

    def link(p, q, rad=0.0):
        ax.annotate("", xy=q, xytext=p, arrowprops=dict(
            arrowstyle="->", lw=1, color="#0072B2", shrinkA=14, shrinkB=14,
            connectionstyle=f"arc3,rad={rad}"))

    cloud(0.6, 5.5); cloud(11.4, 5.5); cloud(0.6, 1.2)
    stock(6, 5.5, "Waiting list"); stock(6, 1.2, "Clinicians")
    flow(1.05, 4.9, 5.5, "Referrals", 3)
    flow(7.1, 10.95, 5.5, "Treatments", 9)
    flow(1.05, 4.9, 1.2, "Net hiring", 3)
    var(3, 7.1, "Expected\nreferrals"); var(6, 3.6, "Desired\ntreatments")
    var(3, 3.3, "Desired\nclinicians"); var(9, 3.6, "Capacity")
    var(10.8, 2.1, "Productivity"); var(8.4, 0.5, "Schedule\npressure")
    link((3, 5.5), (3, 7.1)); link((3, 7.1), (6, 3.6), rad=-0.2)
    link((6, 5.5), (6, 3.6)); link((6, 3.6), (3, 3.3))
    link((3, 3.3), (3, 1.2)); link((6, 1.2), (9, 3.6), rad=0.2)
    link((9, 3.6), (9, 5.5)); link((10.8, 2.1), (9, 3.6))
    link((6, 3.6), (8.4, 0.5), rad=0.3); link((8.4, 0.5), (10.8, 2.1))
    link((6.5, 1.2), (8.4, 0.5))
    ax.set_xlim(0, 12); ax.set_ylim(-0.4, 7.8); ax.axis("off")


fig, ax = plt.subplots(figsize=(9, 5.5))
draw_case(ax)
ax.set_title("Waiting list with treatment and staffing rules")
fig.tight_layout()

Figure: Stock-and-flow diagram of the running case at the end of the chapter. Two stocks (Waiting list in patients, Clinicians in clinicians) are linked by decision rules: desired treatments from the list and expected referrals, desired clinicians from desired treatments, and productivity from schedule pressure. Expected referrals is drawn as a variable for clarity; in the model it is a smoothed stock.

**Try it 1.**

- *Check.* In the base model, set `Capacity` to 4,000 patients/month in `run()`. What does the list do?
- *Apply.* Referrals rise by 400 patients/month in month 24 (`referral_increase=400`). What constant capacity would hold the list level after the rise?
- *Extend.* Set `capacity=10000`. The list cannot fall below zero, but where does it settle, and which term in `Treatments` decides that?

<details><summary>Worked answer</summary>

At 4,000 patients/month the flows balance and the list stays at 32,000 patients. After the rise, only a capacity of 4,400 patients/month holds the list level. Set from the start, that capacity first draws the list down to 22,400 patients by month 24, then holds it there. With a capacity of 10,000 patients/month the list falls fast and settles at 2,000 patients: `Treatments` switches to `Waiting_list / Minimum_treatment_time`, and the list settles where that equals referrals, 4,000 × 0.5 = 2,000. The cell below runs all three.
</details>

In [ ]:
print(f"capacity 4000: list at month 120 "
      f"{base.run(capacity=4000)['Waiting list'].iloc[-1]:,.0f}")
d = base.run(capacity=4400, referral_increase=400)
print(f"capacity 4400 after the rise: list at month 24 "
      f"{d['Waiting list'][24]:,.0f}, month 120 "
      f"{d['Waiting list'].iloc[-1]:,.0f}")
d = base.run(capacity=10000)
print(f"capacity 10000: list at month 120 {d['Waiting list'].iloc[-1]:,.0f},"
      f" treatments {d['Treatments'].iloc[-1]:,.0f}")

## 2. Adjusting towards a goal

The commonest decision rule closes a gap between where a stock is and where someone wants it. You met the bare form as the Close Gap molecule in chapter 3. Two refinements make it a rule a manager would actually use.

**First-Order Stock Adjustment** (`stockAdjust`) orders to replace what flows out and, on top of that, to close the gap over an **adjustment time** $\tau$:

$$
\text{Stocking} = \max\!\left(0,\; \frac{L^{*} - L}{\tau} + r \cdot \text{Outflow}\right),
$$

where $L^{*}$ is the desired level and $r$ the replacement fraction (1 to replace, 0 not to).

**Capacity Utilisation** (`capacityUtil`) recognises that output cannot jump to whatever is wanted. Desired production $P^{*}$, divided by capacity $K$, gives an indicated utilisation $u^{*}$, and a curve bends actual utilisation below 1:

$$
u^{*} = \frac{\max(0, P^{*})}{K}, \qquad \text{Production} = K\left(1 - e^{-1.3\,u^{*}}\right).
$$

Both are after Hines, *Molecules of Structure* [@hines_molecules]. Ours protect the stocking rate with a $\max(0,\cdot)$ and write the utilisation curve as a formula rather than a lookup, so it can be read and differentiated; replace it with a `TABLE` when you have evidence for a different shape.

The worked example switches the replacement term off.

In [ ]:
st = lb.molecule("stockAdjust")
fig, ax = plt.subplots()
for label, kw in [("replace and correct", {}),
                  ("correct only", {"replacement_fraction": 0})]:
    level = st.run(**kw)["Level"]
    ax.plot(level.index, level, label=label)
    print(f"{label:<20}: level at month 60 {level.iloc[-1]:5.1f} units")
ax.axhline(100, color="grey", lw=1, ls=":")
ax.set(title="Stock adjustment with and without replacement",
       xlabel="time (month)", ylabel="Level (units)")
ax.legend()
fig.tight_layout()

Figure: First-Order Stock Adjustment, desired level 100 units, use 10 units/month, adjustment time 4 months. With replacement the level reaches 100 units; without it, it settles at 60 units.

Without replacement the only inflow is the gap divided by the adjustment time, so to match a use of 10 units/month the gap must stay at 10 × 4 = 40 units. This **steady-state error** is the mark of a rule that forgot what flows out. The fix is not to adjust faster but to add the expected outflow to the rule.

The waiting list needs the same logic, turned round, because treatments drain the stock the manager wants smaller. The manager's **desired treatments** replace expected referrals and work the list down to a **target list** over a **time to reduce list**:

$$
\text{Desired treatments} = \max\!\left(0,\; \hat{R} + \frac{W - W^{*}}{\tau_W}\right),
$$

where $\hat{R}$ is `Expected referrals`, a smooth of referrals over 3 months (the Smooth molecule from chapter 4), $W$ the list and $W^{*}$ the target. The manager cannot see referrals next month, so the rule reads the smoothed past.

In [ ]:
RULE = [
    S("Expected referrals", "Base_referrals", "patients/month"),
    F("Updating expectation",
      "(Referrals - Expected_referrals) / Smoothing_time",
      "patients/month/month", dst="Expected referrals"),
    V("Smoothing time", "3", "month"),
    V("Target list", "32000", "patients"),
    V("Time to reduce list", "12", "month"),
    V("Desired treatments", "MAX(0, Expected_referrals + "
      "(Waiting_list - Target_list) / Time_to_reduce_list)",
      "patients/month"),
]
rule = build(BASE, RULE)
r = rule.run(referral_increase=400)
fig, ax = plt.subplots()
for col in ["Referrals", "Desired treatments", "Treatments"]:
    ax.plot(r.index, r[col], label=col)
ax.set(title="Desired treatments outrun a fixed capacity",
       xlabel="time (month)", ylabel="rate (patients/month)")
ax.legend()
for t in (24, 60, 120):
    print(f"month {t:>3}: list {r['Waiting list'][t]:7,.0f} patients, "
          f"desired {r['Desired treatments'][t]:5,.0f}, "
          f"treatments {r['Treatments'][t]:5,.0f} patients/month")

Figure: Referrals rise from 4,000 to 4,400 patients/month in month 24. Desired treatments follow the referrals and the growing list, but treatments stay at the fixed capacity of 3,950 patients/month.

The rule is working: it asks for more treatments every month the list grows, reaching 8,100 patients/month by month 120. Nothing yet turns that wish into capacity, so the list grows by 450 patients/month after the rise. Keeping the desired rate and the achieved rate as separate variables is what lets the model show this gap; a model that set `Treatments = Desired treatments` would hide it and claim an outcome no one could deliver.

The family table runs each molecule once with its default settings and one variant.

In [ ]:
def run(key, **kw):
    return lb.molecule(key).run(**kw)


def family(rows):
    """One row per molecule: what it decides, its key equation, a run."""
    return pd.DataFrame(
        [{"molecule": k, "decides": what, "key equation": eq,
          "run result": fn()} for k, what, eq, fn in rows]
    ).set_index("molecule")


family([
    ("stockAdjust", "inflow to hold a stock at a goal",
     "max(0, (L* - L)/tau + r Outflow)",
     lambda: "level {:.0f} with replacement, {:.0f} without".format(
         run("stockAdjust")["Level"].iloc[-1],
         run("stockAdjust", replacement_fraction=0)["Level"].iloc[-1])),
    ("capacityUtil", "output from desired output and capacity",
     "K (1 - exp(-1.3 P*/K))",
     lambda: "inventory {:.1f} (target 100); {:.1f} when demand > "
     "capacity".format(
         run("capacityUtil")["Inventory"].iloc[-1],
         run("capacityUtil", preset="Demand > capacity")["Inventory"]
         .iloc[-1])),
])

Capacity Utilisation ends week 60 at an inventory of 47.9 units, against a target of 100 units, even though capacity (20 units/week) exceeds demand (18 units/week). The utilisation curve only approaches 1, so meeting demand needs a desired production well above demand, and the inventory gap supplies it. When demand (25 units/week) exceeds capacity (15 units/week), inventory falls to 14.9 units.

**Try it 2.**

- *Check.* In `stockAdjust` without replacement, predict the final level for an adjustment time of 2 months, then run it.
- *Apply.* In the case run above, how many clinicians at 20 patients/clinician/month would the month-120 desired treatments need, against the 197.5 clinicians that give 3,950 patients/month?
- *Extend.* Show by hand that `capacityUtil` must settle near 47.9 units: production must equal sales of 18 units/week.

<details><summary>Worked answer</summary>

Without replacement and with an adjustment time of 2 months the gap is 10 × 2 = 20 units, so the level settles at 80 units. The desired treatments of 8,100 patients/month at month 120 would need 405 clinicians, more than twice the staff in post: a target that can no longer be met by any plausible hiring. For `capacityUtil`, utilisation must be 18 / 20 = 0.9, so $1 - e^{-1.3 u^*} = 0.9$ and $u^{*} = \ln 10 / 1.3 = 1.771$. Desired production is then 35.4 units/week, and since it equals demand plus the gap over 3 weeks, the gap is (35.4 − 18) × 3 = 52.3 units: an inventory of 47.7 units. The default run is still creeping down at week 60 (47.9 units); a run of 300 weeks reaches 47.7 units.
</details>

In [ ]:
lev = run("stockAdjust", replacement_fraction=0, adjustment_time=2)["Level"]
print(f"no replacement, adjustment time 2: level {lev.iloc[-1]:.1f}")
print(f"clinicians for month-120 desired treatments: "
      f"{r['Desired treatments'].iloc[-1] / 20:.0f}")
u = np.log(10) / 1.3
print(f"indicated utilisation {u:.3f}, desired production {20 * u:.1f}, "
      f"inventory {100 - (20 * u - 18) * 3:.1f} units")
print(f"run of 300 weeks: inventory "
      f"{run('capacityUtil', T=300)['Inventory'].iloc[-1]:.1f} units")

## 3. Nonlinear effects and TABLE functions

Many decisions are not made by anyone in particular. When the list grows, clinicians see more patients per session, shorten consultations and skip breaks. Nobody signs off on it; it is an **effect**. An effect multiplies a normal rate by a function of a pressure measured relative to its normal value, so it equals 1 under normal conditions.

**Effect of a Dimensionless Input** (`effectFunction`) is the pattern in its plainest form. An input $x$ is divided by a reference $x_0$, and the ratio passes through a curve that is 0 at zero, 1 at the reference and never above 2:

$$
\text{Effect} = \frac{2\,(x/x_0)}{1 + x/x_0}, \qquad \text{Flow} = \text{Normal flow} \times \text{Effect}.
$$

After Hines, *Molecules of Structure* [@hines_molecules]; ours fixes the curve as a formula so that every reader gets the same shape.

Real effects are usually not a neat formula. They are a judgement, from interviews or data, drawn as a curve. The equation language has `TABLE(x, x1, y1, x2, y2, ...)` for this: straight lines between the points, flat beyond the ends. For the running case, **schedule pressure** is desired treatments over what the clinicians could treat at **normal productivity** of 20 patients/clinician/month, and the effect of pressure on productivity is this table:

In [ ]:
X_P = [0, 0.5, 1, 1.5, 2, 3]
Y_P = [0.8, 0.85, 1, 1.12, 1.18, 1.2]
pts = ", ".join(f"{x:g}, {y:g}" for x, y in zip(X_P, Y_P))
EFFECT_EQ = f"TABLE(Schedule_pressure, {pts})"
print(EFFECT_EQ)

PRESSURE = [
    V("Clinicians", "197.5", "clinicians"),
    V("Normal productivity", "20", "patients/clinician/month"),
    V("Schedule pressure",
      "Desired_treatments / MAX(1, Clinicians * Normal_productivity)",
      "dimensionless"),
    V("Effect of pressure on productivity", EFFECT_EQ, "dimensionless"),
    V("Productivity",
      "Normal_productivity * Effect_of_pressure_on_productivity",
      "patients/clinician/month"),
    V("Capacity", "Clinicians * Productivity", "patients/month"),
]

xs = np.linspace(0, 4, 81)
ef = [lb.Model.from_spec({"name": "t", "vars": [
    {"id": "p", "name": "Schedule pressure", "eq": f"{x}"},
    {"id": "e", "name": "Effect", "eq": EFFECT_EQ}]}).run(T=0)["Effect"]
    .iloc[0] for x in xs]
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(xs, ef, label="TABLE effect")
ax.plot(X_P, Y_P, "o", color="black", label="table points")
ax.axhline(1, color="grey", lw=1, ls=":"); ax.axvline(1, color="grey", lw=1,
                                                     ls=":")
ax.set(title="Effect of schedule pressure on productivity",
       xlabel="schedule pressure (dimensionless)",
       ylabel="effect on productivity (dimensionless)")
ax.legend()
fig.tight_layout()

Figure: The table effect of schedule pressure on productivity. It passes through (1, 1), rises by at most 20% at high pressure and falls to 0.8 when there is little work; beyond the last point it stays flat.

Each feature of the curve is a claim you should be able to defend:

1. **It passes through (1, 1).** At normal pressure the model runs at normal productivity, so the effect does not shift the base case.
2. **Its slope near 1 is modest.** Moving from a pressure of 1 to 1.5 raises productivity by 12%: clinicians can work harder for a while, not double their output.
3. **It saturates.** Beyond a pressure of 3 the effect stays at 1.2. However long the list, a clinic session has only so many slots.
4. **It falls below 1 when work is short.** With little to do, work expands to fill the time.

The cell below adds pressure to the case. The clinicians are still a fixed 197.5, but their productivity now responds.

In [ ]:
pressured = build(BASE, RULE, PRESSURE)
flat = pressured.with_equation("Effect of pressure on productivity", "1")
fig, ax = plt.subplots()
for label, m in [("productivity responds", pressured),
                 ("no effect (always 1)", flat)]:
    d = m.run(referral_increase=400)
    ax.plot(d.index, d["Waiting list"], label=label)
    print(f"{label:<22}: list at month 120 {d['Waiting list'].iloc[-1]:7,.0f}"
          f" patients, productivity {d['Productivity'].iloc[-1]:5.2f}, "
          f"pressure {d['Schedule pressure'].iloc[-1]:4.2f}")
ax.set(title="Pressure on productivity slows the growth of the list",
       xlabel="time (month)", ylabel="Waiting list (patients)")
ax.legend()
fig.tight_layout()

Figure: Waiting list after a 10% rise in referrals in month 24, with 197.5 clinicians throughout. When productivity responds to pressure, the list grows far less than when it does not.

The effect does real work. Without it the list climbs to 76,400 patients by month 120. With it, clinicians lift their productivity to about 22 patients/clinician/month and the list ends near 46,750 patients. The pressure effect is a balancing loop: a longer list raises pressure, which raises treatments, which shortens the list. It is also a warning. The extra 10% comes from people working harder for years on end; chapter 7, Patterns that repeat, shows how that effort can wear off as fatigue builds.

The family table for effects holds one molecule, `effectFunction`; the case's `TABLE` effect is its applied form.

In [ ]:
family([
    ("effectFunction", "multiplier on a normal flow from a relative input",
     "2 (x/x0) / (1 + x/x0)",
     lambda: "Result {:.0f} at reference, {:.0f} at 2x, {:.0f} at 0.4x"
     .format(*[run("effectFunction", input=i)["Result"].iloc[-1]
               for i in (100, 200, 40)])),
])

At the reference input the `Result` stock settles at 120 units; doubling the input raises it only to 160 units, because the effect at twice the reference is 4/3, not 2.

**Try it 3.**

- *Check.* What effect does `effectFunction` give at ten times the reference (`input=1000`)?
- *Apply.* Read the case table by hand at a schedule pressure of 1.25, then confirm it with a one-variable model.
- *Extend.* Replace the table with a straight line, `1 + 0.4 * (Schedule_pressure - 1)`, which is close to the table near 1 but never saturates. Compare the list and productivity at month 120 for the 10% rise, then for a 50% rise (`referral_increase=2000`). Which formulation would you defend to a clinical director?

<details><summary>Worked answer</summary>

At ten times the reference the effect is 2 × 10 / 11 = 1.818. At a pressure of 1.25 the table interpolates halfway between (1, 1) and (1.5, 1.12), giving 1.06. For the 10% rise the straight line ends with a shorter list (about 39,800 against 46,750 patients) and slightly higher productivity (22.2 patients/clinician/month). For the 50% rise the two part company: the table holds productivity at its ceiling of 24.0, while the line lets it climb to 30.2 patients/clinician/month, 51% above normal, and so ends with a list of about 67,500 patients instead of 158,800. The line promises output that no clinic could sustain. The table is the one to defend: it states a limit that clinicians and managers can check.
</details>

In [ ]:
print(f"effect at 10x reference: "
      f"{run('effectFunction', input=1000)['Effect'].iloc[-1]:.3f}")
probe = lb.Model.from_spec({"name": "probe", "vars": [
    {"id": "p", "name": "Schedule pressure", "eq": "1.25"},
    {"id": "e", "name": "Effect", "eq": EFFECT_EQ}]})
print(f"table at pressure 1.25: {probe.run(T=0)['Effect'].iloc[0]:.3f}")
line = pressured.with_equation("Effect of pressure on productivity",
                               "1 + 0.4 * (Schedule_pressure - 1)")
for rise in (400, 2000):
    for label, m in [("table", pressured), ("line", line)]:
        d = m.run(referral_increase=rise)
        print(f"rise {rise}, {label:<5}: list at month 120 "
              f"{d['Waiting list'].iloc[-1]:7,.0f}, productivity "
              f"{d['Productivity'].iloc[-1]:5.2f}")

## 4. Anchors: how expectations and judgements form

Managers rarely work a number out from first principles. Next year's budget starts from this year's; a target list starts from last year's list. **Anchoring and adjustment** writes that habit down: a value is an anchor multiplied by adjustments, each equal to 1 when its cue is normal.

$$
\begin{aligned}
\text{Univariate:}\quad & V = A \left(\frac{c}{c_0}\right)^{s} \\
\text{Multivariate:}\quad & V = A \prod_i f_i(p_i), \quad f_i(1) = 1 \\
\text{Sea anchor:}\quad & V = A\,p, \qquad \frac{dA}{dt} = \frac{V - A}{\tau_A} \\
\text{Protected sea anchor:}\quad & \frac{dA}{dt} = \frac{(1-\pi)\,A\,p + \pi\,F - A}{\tau_A} \\
\text{Present value:}\quad & \frac{d\,PV}{dt} = C\,e^{-\rho t}
\end{aligned}
$$

In **Univariate Anchoring and Adjustment** (`univariateAnchor`) the strength $s$ sets how far the judgement moves: 1 is proportional, 0.5 a square-root under-adjustment. **Multivariate Anchoring and Adjustment** (`multivariateAnchor`) multiplies several such adjustments. In **Sea Anchor and Adjustment** (`seaAnchor`) the anchor itself drifts towards the value it produced: a weight that is dragged along behind a boat, hence the name. **Protected Sea Anchor and Adjustment** (`protectedSeaAnchor`) ties a share $\pi$ of the anchor's target to a fundamental $F$. **Present Value** (`presentValue`) is the special case where the judgement is about the future: a cash flow $C$ discounted at rate $\rho$ (a percent per year in the molecule) and accumulated.

All five are after Hines, *Molecules of Structure* [@hines_molecules]. Ours write each adjustment as a power or a clipped formula, where Hines leaves the shape to a lookup, and our present value uses continuous discounting.

The worked example is the sea anchor, because it is the one that bites in the running case. Suppose the target list is set each year at last year's list, adjusted for pressure. If the pressure says "a little more is acceptable", the target follows the list up.

In [ ]:
fig, ax = plt.subplots()
for label, kw in [("pressure 1.00", {}),
                  ("pressure 1.02", {"adjustment_pressure": 1.02}),
                  ("pressure 1.20", {"adjustment_pressure": 1.2}),
                  ("pressure 1.20, protected 50%",
                   {"molecule": "protectedSeaAnchor"})]:
    key = kw.pop("molecule", "seaAnchor")
    a = run(key, **kw)["Anchor"]
    ax.plot(a.index, a, label=label)
    print(f"{label:<30}: anchor at month 60 {a.iloc[-1]:7.1f}")
ax.set_yscale("log")
ax.set(title="A sea anchor drifts; a protected one settles",
       xlabel="time (month)", ylabel="Anchor (units, log scale)")
ax.legend()
fig.tight_layout()

Figure: Sea Anchor and Adjustment with anchor lag 6 months. At a pressure of exactly 1 the anchor stays at 100 units. Any persistent pressure above 1 makes it grow without limit; protecting half of the anchor's target with a fundamental of 100 units makes it settle at 125 units.

A pressure of only 1.02 lifts the anchor to 122 units in 60 months, and 1.2 to 736 units. The anchor chases the value it produced, so the structure is a reinforcing loop with no end. This is how targets drift: a target list anchored on the last list, adjusted upward "to be realistic", rises every year without anyone ever deciding to accept a longer wait. Protection fixes it by giving the anchor something outside itself to hold on to, here a national standard. With half protection the equilibrium solves $A = 0.5 \times 1.2 A + 0.5 \times 100$, so $A = 125$.

In [ ]:
family([
    ("univariateAnchor", "a judgement from one cue",
     "A (c/c0)^s",
     lambda: "value {:.1f} at s = 0.5, {:.1f} at s = 1 (cue 1.5x)".format(
         run("univariateAnchor")["Value in use"].iloc[-1],
         run("univariateAnchor", adjustment_strength=1)["Value in use"]
         .iloc[-1])),
    ("multivariateAnchor", "a judgement from several cues",
     "A f1(p1) f2(p2) f3(p3)",
     lambda: "value {:.0f} (pressures 2.25, 1, 1.2); {:.0f} when two "
     "offset".format(
         run("multivariateAnchor")["Value"].iloc[-1],
         run("multivariateAnchor", preset="Offsetting pressures")["Value"]
         .iloc[-1])),
    ("seaAnchor", "a judgement whose anchor follows it",
     "dA/dt = (A p - A)/tau",
     lambda: "anchor {:.0f} at p = 1, {:.0f} at p = 1.2".format(
         run("seaAnchor")["Anchor"].iloc[-1],
         run("seaAnchor", adjustment_pressure=1.2)["Anchor"].iloc[-1])),
    ("protectedSeaAnchor", "the same, tied to a fundamental",
     "target (1-pi) A p + pi F",
     lambda: "anchor {:.1f} with pi = 0.5; {:.0f} with pi = 0".format(
         run("protectedSeaAnchor")["Anchor"].iloc[-1],
         run("protectedSeaAnchor", protection=0)["Anchor"].iloc[-1])),
    ("presentValue", "today's worth of a future stream",
     "dPV/dt = C exp(-rho t)",
     lambda: "PV {:.0f} at 8%, {:.0f} at 4% (20 a year, 60 years)".format(
         run("presentValue")["Present value"].iloc[-1],
         run("presentValue", discount_rate=4)["Present value"].iloc[-1])),
])

Two readings from the table. A cue 50% above normal moves an anchored judgement only to 122.5 units when the strength is 0.5, against 150 units at full strength. And a cash flow of 20 a year for 60 years is worth 249 today at 8% a year, but 456 at 4%: the discount rate decides whether a long-lived investment, such as a new clinic, looks worth it.

**Try it 4.**

- *Check.* With strength 0.5, what cue (as a multiple of normal) moves `univariateAnchor` to 150 units?
- *Apply.* With `seaAnchor` at a pressure of 1.02, in which month does the anchor first pass 110 units?
- *Extend.* In `protectedSeaAnchor` at pressure 1.2, what protection keeps the anchor at or below 150 units in the long run? Derive it, then check with a run of 600 months.

<details><summary>Worked answer</summary>

Reaching 150 needs $(c/c_0)^{0.5} = 1.5$, so a cue of 2.25 times normal (225 units against a normal 100). At a pressure of 1.02 the anchor grows about 0.33% a month, passing 110 units in month 28.6. In equilibrium $A = \pi F / (1 - 1.2(1 - \pi))$; setting $A = 150$ and $F = 100$ gives $\pi = 0.375$. The long run settles at 150.0 units with that protection.
</details>

In [ ]:
v = run("univariateAnchor", current_value_of_variable=225)["Value"]
print(f"cue 225: value {v.iloc[-1]:.1f}")
a = run("seaAnchor", adjustment_pressure=1.02)["Anchor"]
print(f"anchor first above 110 in month {a.index[a > 110][0]:.1f}")
a = run("protectedSeaAnchor", protection=0.375, T=600)["Anchor"]
print(f"protection 0.375: anchor after 600 months {a.iloc[-1]:.1f}")

## 5. Allocating a scarce resource

A hospital group rarely has one waiting list. Clinicians are shared between specialties, and someone decides each week who gets the sessions. Split rules write that decision. Each claimant $i$ has a **claim** $c_i$ (often its own backlog), optionally a managerial **weight** $w_i$, and receives a share of the resource $R$:

$$
R_i = R\,\frac{w_i c_i}{\sum_j w_j c_j}, \qquad \text{Fraction to A} = \min\!\left(1 - m,\; \max\!\left(m,\; \frac{c_A}{c_A + c_B}\right)\right).
$$

**Proportional Split** (`propSplit`) is the left formula with every weight 1. **Weighted Split** (`weightedSplit`) adds the weights. **Multidimensional Split** (`multiSplit`) divides a flow by fixed weights, with no claim from the receiving stocks. **Nonlinear Split** (`nonlinearSplit`) is the right formula: a minimum share $m$ that each side always gets. All four are after Hines, *Molecules of Structure* [@hines_molecules]; ours guard each share against a zero total claim and use a clamp where Hines uses a lookup for the nonlinear split.

Worked example: three clinics (A, B and C) share 12 clinicians, each clinic gets 4 new referrals a week, and the manager weights clinic A twice, say because its patients are more urgent.

In [ ]:
ws = lb.molecule("weightedSplit")
fig, ax = plt.subplots()
for label, kw in [("weight for A = 2", {}), ("equal weights", {
        "weight_for_a": 1})]:
    d = ws.run(**kw)
    ax.plot(d.index, d["Tasks A"], label=f"Tasks A, {label}")
    ax.plot(d.index, d["Tasks B"], label=f"Tasks B, {label}")
    print(f"{label:<16}: A {d['Tasks A'].iloc[-1]:5.1f}, B "
          f"{d['Tasks B'].iloc[-1]:5.1f}, C {d['Tasks C'].iloc[-1]:5.1f} "
          f"tasks; resources to A {d['Resources to A'].iloc[-1]:.2f} people")
ax.set(title="A managerial weight shortens one backlog",
       xlabel="time (week)", ylabel="backlog (tasks)")
ax.legend()
fig.tight_layout()

Figure: Weighted Split with 12 people across three backlogs, each receiving 4 tasks/week, starting at 30 tasks each. Weighting A twice moves its backlog from 30 to 18 tasks, and B's and C's up to 36 tasks.

The weight does not change how many people A gets in the end: all three clinics settle at 4 people, which is what their arrivals need. It changes the backlogs at which that balance is struck. In equilibrium every weighted claim is equal, $2 T_A = T_B = T_C$, and because total work stays at 90 tasks, $T_A = 18$ tasks. Priority buys a shorter queue for one clinic at the expense of the others, not more output.

In [ ]:
family([
    ("propSplit", "share a resource by backlog size",
     "R c_i / sum c",
     lambda: "Tasks A {:.0f} when capacity = arrivals; {:.0f} when "
     "overloaded".format(
         run("propSplit")["Tasks A"].iloc[-1],
         run("propSplit", resources=7)["Tasks A"].iloc[-1])),
    ("weightedSplit", "the same, tilted by priorities",
     "R w_i c_i / sum w c",
     lambda: "Tasks A {:.0f} with weight 2, {:.0f} with weight 1".format(
         run("weightedSplit")["Tasks A"].iloc[-1],
         run("weightedSplit", weight_for_a=1)["Tasks A"].iloc[-1])),
    ("multiSplit", "divide a flow by fixed weights",
     "Inflow w_i / sum w",
     lambda: "channels {:.1f}, {:.1f}, {:.1f} (weights 5 : 3 : 2)".format(
         *[run("multiSplit")[c].iloc[-1]
           for c in ("Channel A", "Channel B", "Channel C")])),
    ("nonlinearSplit", "a share with a guaranteed minimum",
     "min(1-m, max(m, c_A/sum c))",
     lambda: "B holds {:.0f} with m = 0.2, {:.0f} with m = 0".format(
         run("nonlinearSplit")["Held by B"].iloc[-1],
         run("nonlinearSplit", minimum_fraction=0)["Held by B"].iloc[-1])),
])

In `propSplit`, when 7 people face 10 tasks/week of arrivals, every backlog grows, and the largest (A, with 5 tasks/week) grows most, to 137 tasks by week 60. In `nonlinearSplit` the minimum share of 20% doubles what claimant B holds, from 10 to 20 units, even though its claim is only a tenth of the total.

**Try it 5.**

- *Check.* In `multiSplit`, why does Channel A settle at 37.5 units?
- *Apply.* What weight for A brings its backlog in `weightedSplit` down to 15 tasks? Derive it from the equilibrium, then run it.
- *Extend.* In `nonlinearSplit` with a minimum fraction of 0.3, how much does B hold, and what does that say about a "fair share" rule when one claim is far stronger?

<details><summary>Worked answer</summary>

Channel A receives 15 × 5/10 = 7.5 units/month and drains over 5 months, so it settles at 7.5 × 5 = 37.5 units. For `weightedSplit`, the equilibrium needs $w_A T_A = T_B = T_C$ with $T_A + T_B + T_C = 90$, so $T_A (1 + 2 w_A) = 90$ and $w_A = 2.5$ for 15 tasks; the run gives 15.0 tasks. With a minimum of 0.3 the fraction to A is capped at 0.7, so B receives 6 units/week and holds 30 units: a fair-share floor can hand a weak claimant three times what its claim would earn.
</details>

In [ ]:
print(f"weight 2.5: Tasks A "
      f"{run('weightedSplit', weight_for_a=2.5)['Tasks A'].iloc[-1]:.1f}")
hb = run("nonlinearSplit", minimum_fraction=0.3)["Held by B"]
print(f"minimum fraction 0.3: Held by B {hb.iloc[-1]:.1f}")

## 6. Protecting flows and backlogs

A decision rule can ask for anything; a stock can only give what it has. Four molecules make outflows respect that, and the running case has used one since section 1. In each, $D$ is the outflow someone would like, $L$ the stock it drains, $\tau_{\min}$ the shortest time in which the stock could physically be emptied, and $B$ a backlog.

$$
\begin{aligned}
\text{Protected by flow:}\quad & \text{Outflow} = \min\!\left(D,\; L / \tau_{\min}\right) \\
\text{Protected by level:}\quad & \text{Outflow} = D \cdot \min\!\left(1, \max\!\left(0, L / L_{\text{full}}\right)\right) \\
\text{Backlog by flow:}\quad & \text{Shipping} = \min\!\left(B / \tau_B,\; K\right) \\
\text{Backlog by level:}\quad & \text{Shipping} = \frac{B}{\tau_B} \cdot \min\!\left(1, \max\!\left(0, I / I_{\text{full}}\right)\right)
\end{aligned}
$$

These are **Level Protected by Flow** (`protFlow`), **Level Protected by Level** (`protLevel`), **Backlog Shipping Protected by Flow** (`backlogFlow`) and **Backlog Shipping Protected by Level** (`backlogLevel`), all after Hines, *Molecules of Structure* [@hines_molecules]. Ours use a minimum draining time with a physical meaning instead of the simulation's time step, and a clipped straight line where Hines uses a smooth lookup for the level effects.

The case's `Treatments = MIN(Capacity, Waiting_list / Minimum_treatment_time)` is Backlog Shipping Protected by Flow: the list is the backlog, half a month is the fastest it could be cleared. The worked example tests it at an extreme: referrals stop entirely, with ample capacity. A naive rule, `Treatments = Capacity`, is the comparison.

In [ ]:
extreme = dict(base_referrals=0, capacity=6000)
naive = base.with_equation("Treatments", "Capacity")
fig, ax = plt.subplots()
for label, m in [("protected: MIN(Capacity, list / 0.5)", base),
                 ("naive: Treatments = Capacity", naive)]:
    d = m.run(T=12, warn_clips=False, **extreme)
    ax.plot(d.index, d["Waiting list"], label=label)
    first = d.attrs["clipped"].get("Waiting list")
    print(f"{label:<38}: list at month 12 {d['Waiting list'].iloc[-1]:7.1f},"
          f" treatments {d['Treatments'].iloc[-1]:7.1f}, floor held from "
          f"month {first}")
ax.axhline(0, color="grey", lw=1)
ax.set(title="Extreme test: referrals stop",
       xlabel="time (month)", ylabel="Waiting list (patients)")
ax.legend()
fig.tight_layout()

Figure: Waiting list when referrals fall to zero and capacity is 6,000 patients/month. The protected rule empties the list smoothly; the naive rule drives it to zero in about five and a half months, after which only the engine's floor at zero holds it there.

The protected rule never asks for more than the list holds, so treatments fall towards zero as the list empties. The naive rule keeps treating 6,000 patients/month from an empty list. The package's floor at zero hides the impossible stock, but `df.attrs["clipped"]` records when it acted (month 5.375 here), and the flow is still wrong: anything that uses `Treatments`, such as cost or staff time, would be counted for patients who do not exist. Write the protection into the equation; treat a clip as a bug report.

In [ ]:
family([
    ("protFlow", "an outflow capped by what the stock holds",
     "min(D, L / tau_min)",
     lambda: "settles at {:.0f} units (3 in, wanted 10 out)".format(
         run("protFlow")["On hand"].iloc[-1])),
    ("protLevel", "an outflow that fades as the stock empties",
     "D min(1, L / L_full)",
     lambda: "settles at {:.2f} units (4 in, wanted 12 out)".format(
         run("protLevel")["Inventory"].iloc[-1])),
    ("backlogFlow", "shipping capped by capacity",
     "min(B / tau_B, K)",
     lambda: "backlog {:.0f} with capacity 10; {:.0f} with 20".format(
         run("backlogFlow")["Backlog"].iloc[-1],
         run("backlogFlow", shipping_capacity=20)["Backlog"].iloc[-1])),
    ("backlogLevel", "shipping limited by inventory on hand",
     "B / tau_B x min(1, I / I_full)",
     lambda: "backlog {:.0f}, inventory {:.0f}; starved: backlog "
     "{:.0f}, inventory {:.1f}".format(
         run("backlogLevel")["Backlog"].iloc[-1],
         run("backlogLevel")["Inventory"].iloc[-1],
         run("backlogLevel", production=5)["Backlog"].iloc[-1],
         run("backlogLevel", production=5)["Inventory"].iloc[-1])),
])

`backlogFlow` with 12 orders/month against a capacity of 10 grows a backlog of 180 orders by month 60, still climbing; with capacity 20 it settles at 36 orders, three months of orders at the desired ship time of 3 months. `backlogLevel` with production starved to 5 units/month lets the backlog reach 352 orders while inventory sits near empty, at about 1.7 units.

A caution about `backlogLevel`: its protection fades shipping as inventory falls, but within one time step it can still ask for more than the inventory holds when orders far outrun production. Keep the order rate within a few times production, or shorten `dt`, and check `df.attrs["clipped"]` after every run.

**Try it 6.**

- *Check.* In `protFlow`, the *Slow extraction* preset sets the fastest draining time to 5 weeks. Where does the stock settle?
- *Apply.* In `protLevel`, derive the settling level from the equation, then compare with the run.
- *Extend.* Rerun the extreme test with `dt=0.5` instead of 0.125. What happens to the protected rule, and what does that tell you about choosing `Minimum treatment time` relative to `dt`?

<details><summary>Worked answer</summary>

In `protFlow` the outflow in equilibrium is `On hand / 5 = 3`, so the stock settles at 15 units. In `protLevel` the outflow is 12 × Inventory / 25 = 4, so Inventory = 25/3 = 8.33 units. With `dt=0.5`, equal to the minimum treatment time, the list falls by 3,000 patients a step until 2,000 patients remain, and then one step of `list / 0.5` removes all of them: the list jumps to zero instead of tailing off. It still never goes negative and nothing clips, but the shape is an artefact of the step. The minimum time must be several times `dt`; the case uses 0.5 month against `dt = 0.125` month, a factor of 4.
</details>

In [ ]:
print(f"slow extraction: "
      f"{run('protFlow', preset='Slow extraction')['On hand'].iloc[-1]:.1f}")
print(f"protLevel: {run('protLevel')['Inventory'].iloc[-1]:.2f}")
d = base.run(T=12, dt=0.5, **extreme)
print(d["Waiting list"].loc[4:6.5].round(1).to_string())
print("clipped:", d.attrs["clipped"])

## 7. Turning resources into action, and back

Most decisions about people pass through one identity, read in different directions. A resource $N$ (clinicians, pumps, money) with an **ability** $a$ (productivity) creates a flow $Q$; the flow wanted tells you the resources needed; an observed flow tells you the ability; and resources cost money.

$$
\begin{aligned}
\text{Action from resource:}\quad & Q = N a \\
\text{Resources from action:}\quad & N^{*} = Q^{*} / a \\
\text{Ability from action:}\quad & \hat{a} = Q / N \\
\text{Financial flow:}\quad & \text{Spending} = N w \\
\text{Workforce from budget:}\quad & N_{\text{aff}} = \text{Budget} / w
\end{aligned}
$$

These are **Action from Resource** (`actionFromResource`), **Resources from Action** (`resourcesFromAction`), **Ability from Action** (`abilityFromAction`), **Financial Flow from Resource** (`financialFlow`) and **Workforce from Budget** (`workforceFromBudget`), after Hines, *Molecules of Structure* [@hines_molecules]. Ours guard each division against a zero denominator and, in `abilityFromAction`, smooth the measured ability over a measuring time so the estimate is a belief, not an instant fact.

The running case now gets its staffing rule. Clinicians become a stock. Desired clinicians are desired treatments divided by normal productivity (resources from action), capped by what the staff budget can pay for (workforce from budget). Net hiring closes the gap over a **time to hire** of 6 months, the Close Gap molecule from chapter 3. Capacity is clinicians times productivity (action from resource).

In [ ]:
STAFF = [
    S("Clinicians", "197.5", "clinicians"),
    F("Net hiring", "(Desired_clinicians - Clinicians) / Time_to_hire",
      "clinicians/month", dst="Clinicians"),
    V("Time to hire", "6", "month"),
    V("Staff budget", "2400000", "euro/month"),
    V("Cost per clinician", "10000", "euro/clinician/month"),
    V("Affordable clinicians", "Staff_budget / Cost_per_clinician",
      "clinicians"),
    V("Desired clinicians", "MIN(Affordable_clinicians, "
      "Desired_treatments / Normal_productivity)", "clinicians"),
    V("Staff cost", "Clinicians * Cost_per_clinician", "euro/month"),
]
staffed = build(BASE, RULE, PRESSURE, STAFF)
print(staffed.equations())

Three versions of the manager's rule, all facing the same 10% rise in referrals in month 24:

- **no hiring:** the section 3 model, staff fixed at 197.5 clinicians and productivity absorbing the pressure;
- **replace only:** hire for expected referrals but ignore the list (`Desired treatments = Expected_referrals`), the steady-state error of section 2 in clinical form;
- **replace and correct:** the full rule, which also works the list back to its target of 32,000 patients over 12 months.

In [ ]:
RULES = {
    "no hiring": pressured,
    "replace only": staffed.with_equation("Desired treatments",
                                          "Expected_referrals"),
    "replace and correct": staffed,
}
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
summary = []
for label, m in RULES.items():
    d = m.run(referral_increase=400)
    ax1.plot(d.index, d["Waiting list"], label=label)
    ax2.plot(d.index, d["Productivity"], label=label)
    summary.append({"rule": label,
                    "peak list (patients)": round(d["Waiting list"].max()),
                    "list at month 120": round(d["Waiting list"].iloc[-1]),
                    "clinicians at 120": round(d["Clinicians"].iloc[-1], 1),
                    "productivity at 120": round(d["Productivity"].iloc[-1],
                                                 2)})
ax1.set(title="Waiting list under three staffing rules",
        xlabel="time (month)", ylabel="Waiting list (patients)")
ax2.set(title="Productivity under the same rules",
        xlabel="time (month)",
        ylabel="productivity (patients/clinician/month)")
ax1.legend()
pd.DataFrame(summary).set_index("rule")

Figure: Left, the waiting list after a 10% rise in referrals in month 24 under three rules. Right, clinical productivity. Hiring brings the list back; hiring that ignores the list leaves it stuck high; no hiring leaves clinicians working at about 10% above normal for good.

The full rule ends with 220 clinicians, brings the list back to its target of 32,000 patients and lets productivity return to normal. Its peak list is about 33,800 patients. Replace-only hiring also ends with 220 clinicians at normal productivity, but the list stays at the level it reached while staff were being hired, about 35,250 patients. Nothing in that rule ever asks for the extra treatments needed to bring it down. No hiring leaves the list at 46,700 patients and still rising, with clinicians at 22 patients/clinician/month. Only the full rule both restores the list and relieves the staff.

In [ ]:
family([
    ("actionFromResource", "a flow from resources and their ability",
     "Q = N a",
     lambda: "flow {:.0f}/month, tank {:.0f} (10 pumps x 3)".format(
         run("actionFromResource")["Flow"].iloc[-1],
         run("actionFromResource")["Tank"].iloc[-1])),
    ("resourcesFromAction", "resources needed for a desired flow",
     "N* = Q* / a",
     lambda: "resources {:.1f} (flow 60, ability 3); {:.1f} at ability "
     "6".format(
         run("resourcesFromAction")["Resources"].iloc[-1],
         run("resourcesFromAction", resource_ability_to_create_flow=6)
         ["Resources"].iloc[-1])),
    ("abilityFromAction", "ability inferred from flow and resources",
     "a_hat = Q / N, smoothed",
     lambda: "measured ability {:.2f} (60 / 10); {:.2f} with 20 "
     "resources".format(
         run("abilityFromAction")["Measured ability"].iloc[-1],
         run("abilityFromAction", resource=20)["Measured ability"]
         .iloc[-1])),
    ("financialFlow", "spending from resources and their rent",
     "Spending = N w",
     lambda: "cash {:.0f} at month 60 (spends 100, earns 80); "
     "{:.0f} at break-even".format(
         run("financialFlow")["Cash"].iloc[-1],
         run("financialFlow", workers=16)["Cash"].iloc[-1])),
    ("workforceFromBudget", "people a budget can pay for",
     "N_aff = Budget / w",
     lambda: "workforce {:.1f}; {:.1f} after a budget cut to 900".format(
         run("workforceFromBudget")["Workforce"].iloc[-1],
         run("workforceFromBudget", workforce_budget=900)["Workforce"]
         .iloc[-1])),
])

`financialFlow` makes a point the other four do not: cash has no floor in the molecule, so a team spending 100 a month against revenue of 80 ends at a cash balance of minus 700 after 60 months. Money can go negative (an overdraft); people cannot, which is why `Workforce` and `Clinicians` are protected stocks and `Cash` is not.

**Try it 7.**

- *Check.* In `resourcesFromAction`, double the target flow to 120. How many resources are needed?
- *Apply.* What monthly staff budget would the full rule need to reach its 220 clinicians at 10,000 euro/clinician/month?
- *Extend.* Cut the staff budget to 2,100,000 euro/month (210 clinicians). Where does the list end, and what productivity are clinicians held at?

<details><summary>Worked answer</summary>

At a target of 120 the molecule needs 40 resources. The full rule's 220 clinicians cost 220 × 10,000 = 2,200,000 euro/month; while it works the list down it briefly overshoots, and its peak staff cost is about 2,245,000 euro/month. Both are under the budget of 2,400,000, so the cap never binds in the base run. With a budget of 2,100,000 euro/month the staff stop at 210 clinicians. Their 4,200 patients/month at normal productivity cannot match 4,400 referrals/month, so pressure stays up: productivity is still at 20.87 patients/clinician/month at month 120, and the list ends at about 38,700 patients, rising by about 18 patients/month. A budget cap turns a staffing problem back into a pressure problem.
</details>

In [ ]:
res = run("resourcesFromAction", desired_flow=120)["Resources"]
print(f"target 120: resources {res.iloc[-1]:.1f}")
d = staffed.run(referral_increase=400)
print(f"full rule: peak staff cost {d['Staff cost'].max():,.0f} euro/month")
d = staffed.run(referral_increase=400, staff_budget=2100000)
print(f"budget 2.1 million: clinicians {d['Clinicians'].iloc[-1]:.1f}, "
      f"productivity {d['Productivity'].iloc[-1]:.2f}, list "
      f"{d['Waiting list'].iloc[-1]:,.0f}, growing "
      f"{d['Waiting list'].diff().iloc[-1] / 0.125:.0f} patients/month")

## 8. Doing work and productivity

The last family describes the work itself. Workers $N$ at productivity $p$ drain a backlog of work; productivity is a normal value multiplied by effects; and managers estimate productivity from what they see done.

$$
\begin{aligned}
\text{Producing:}\quad & \text{Output} = N p \\
\text{Doing work:}\quad & \text{Work done} = \min\!\left(N p,\; B / 0.5\right) \\
\text{Productivity:}\quad & p = p_0 \, f_{\text{fatigue}} \, f_{\text{pressure}} \, f_{\text{adequacy}} \, f_{\text{skill}} \\
\text{Estimated productivity:}\quad & \frac{d\hat{p}}{dt} = \frac{\text{Work done}/N - \hat{p}}{\tau_p}
\end{aligned}
$$

These are **Producing** (`producing`), **Doing Work** (`doingWork`), **Productivity** (`productivity`) and **Estimated Productivity** (`estProductivity`), after Hines, *Molecules of Structure* [@hines_molecules]. Ours write each productivity effect as a clipped straight line rather than a lookup, and protect the work rate with a half-week minimum completion time.

The worked example is the estimate, because it is what a manager actually uses. Ten workers, each truly able to do 5 tasks a month, work through a fixed pile of 1,500 tasks. The manager's perceived productivity starts at a pessimistic 2 tasks/person/month and learns from output over a perception time of 3 months.

In [ ]:
ep = lb.molecule("estProductivity")
d = ep.run()
fig, ax = plt.subplots()
ax.plot(d.index, d["Perceived PDY"], label="perceived productivity")
ax.plot(d.index, d["Estimated PDY"], label="output per worker")
ax.axhline(5, color="grey", lw=1, ls=":", label="actual productivity")
ax.set(title="Productivity estimated from output",
       xlabel="time (month)", ylabel="productivity (tasks/person/month)")
ax.legend()
done = d.index[d["Work to do"] <= 1][0]
print(f"perceived at month 10: {d['Perceived PDY'][10]:.2f}; work runs out "
      f"in month {done:.1f}; perceived at month 40: "
      f"{d['Perceived PDY'][40]:.2f} tasks/person/month")

Figure: Estimated Productivity with 10 workers of true productivity 5 tasks/person/month and 1,500 tasks to do. The estimate learns the truth within about 10 months, then collapses when the work runs out, although nobody became less able.

While work is plentiful the estimate is good: 4.90 tasks/person/month by month 10. Once the pile is gone, output per worker falls to zero and the estimate follows, to 0.17 tasks/person/month by month 40. Output divided by workforce measures productivity only when work is not the constraint. In a clinic, a quiet month looks like slow clinicians, and a busy month like fast ones.

In [ ]:
family([
    ("producing", "output from workers and productivity",
     "Output = N p",
     lambda: "{:.0f} drawings in 60 months (10 x 2/month)".format(
         run("producing")["Drawings produced"].iloc[-1])),
    ("doingWork", "work done from a backlog",
     "min(N p, B / 0.5)",
     lambda: "backlog {:.0f} after clearing; {:.0f} when it grows".format(
         run("doingWork")["Backlog"].iloc[-1],
         run("doingWork", preset="Backlog grows")["Backlog"].iloc[-1])),
    ("productivity", "productivity from its conditions",
     "p0 f1 f2 f3 f4",
     lambda: "{:.2f} tired, pushed, less skilled; {:.2f} normal".format(
         run("productivity", T=0)["Productivity"].iloc[0],
         run("productivity", T=0, preset="All normal")["Productivity"]
         .iloc[0])),
    ("estProductivity", "a manager's belief about productivity",
     "smooth of Work done / N",
     lambda: "perceived {:.2f} at month 10, {:.2f} at month 40".format(
         run("estProductivity")["Perceived PDY"][10],
         run("estProductivity")["Perceived PDY"][40])),
])

In `productivity`, the four effects multiply: fatigue (0.85), schedule pressure (1.08) and lower skill (0.88) together give 4.04 tasks/person/month against a normal 5. Pressure alone would have raised it; fatigue and skill more than cancel the gain. The running case so far has only the pressure effect; chapter 7, Patterns that repeat, adds fatigue.

**Try it 8.**

- *Check.* In `productivity`, what does the *Short of work* preset (work adequacy 0.4) give?
- *Apply.* In `doingWork` with the *Clear backlog* preset (20 workers at 4 tasks/week, no new work), when is the backlog of 600 tasks gone?
- *Extend.* In `estProductivity`, double the workforce to 20. When does the work run out, and what is the perceived productivity at month 20? Which way would a manager reading it misjudge?

<details><summary>Worked answer</summary>

Short of work gives 5 × 0.4 = 2.00 tasks/person/month. The backlog of 600 tasks at 80 tasks/week would be gone in 7.5 weeks at full rate, but the protected rate tails off over the last tasks, so it falls below one task only in week 8.7. With 20 workers the pile runs out in month 16.3, and by month 20 the perceived productivity has fallen to 0.93 tasks/person/month. A manager reading that number would think the larger team a fifth as able as it is, when in truth it simply ran out of work.
</details>

In [ ]:
p = run("productivity", T=0, preset="Short of work")["Productivity"]
print(f"short of work: {p.iloc[0]:.2f}")
d = run("doingWork")
print(f"backlog below 1 task in week {d.index[d['Backlog'] < 1][0]:.1f}")
d = run("estProductivity", workforce=20)
print(f"20 workers: work runs out in month "
      f"{d.index[d['Work to do'] <= 1][0]:.1f}; perceived at month 20 "
      f"{d['Perceived PDY'][20]:.2f}")

## 9. The running case: staffing to measured productivity

Section 7 planned staff on normal productivity, a number the manager was given. In practice a manager may read productivity from last quarter's treatments per clinician, as in section 8. Under pressure that number is inflated, because clinicians are working harder than normal. The cell below adds a perceived productivity (Ability from Action, smoothed over 6 months) and a switch, `Use measured productivity`, that makes the staffing rule plan on it.

In [ ]:
MEASURE = [
    S("Perceived productivity", "Productivity", "patients/clinician/month"),
    F("Updating productivity",
      "(Measured_productivity - Perceived_productivity) / Perception_time",
      "patients/clinician/month/month", dst="Perceived productivity"),
    V("Perception time", "6", "month"),
    V("Measured productivity", "Treatments / MAX(1, Clinicians)",
      "patients/clinician/month"),
    V("Use measured productivity", "0", "dimensionless"),
    V("Planning productivity", "Use_measured_productivity * "
      "Perceived_productivity + (1 - Use_measured_productivity) * "
      "Normal_productivity", "patients/clinician/month"),
    V("Desired clinicians", "MIN(Affordable_clinicians, "
      "Desired_treatments / Planning_productivity)", "clinicians"),
]
case = build(BASE, RULE, PRESSURE, STAFF, MEASURE)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
rows = []
for label, use in [("plan on normal productivity", 0),
                   ("plan on measured productivity", 1)]:
    d = case.run(referral_increase=400, use_measured_productivity=use)
    ax1.plot(d.index, d["Waiting list"], label=label)
    ax2.plot(d.index, d["Clinicians"], label=label)
    rows.append({"rule": label,
                 "peak list": round(d["Waiting list"].max()),
                 "month of peak": d["Waiting list"].idxmax(),
                 "clinicians at month 36": round(d["Clinicians"][36], 1),
                 "list at month 120": round(d["Waiting list"].iloc[-1])})
ax1.set(title="Waiting list", xlabel="time (month)",
        ylabel="Waiting list (patients)")
ax2.set(title="Clinicians", xlabel="time (month)",
        ylabel="Clinicians (clinicians)")
ax1.legend()
pd.DataFrame(rows).set_index("rule")

Figure: Waiting list and clinicians after a 10% rise in referrals in month 24, when the staffing rule plans on normal productivity and when it plans on productivity measured from recent treatments. Measured productivity delays hiring and lets the list peak slightly higher.

The result is honest rather than dramatic. Planning on measured productivity hires a little more slowly (219.3 against 220.8 clinicians at month 36) and lets the list peak a little higher and later: about 33,920 patients in month 34.5, against 33,800 in month 33.75. Both rules end in the same place, because as hiring relieves the pressure, measured productivity falls back to normal and the bias disappears. The bias is self-correcting here because the hiring loop eventually removes the pressure that created it. It would not be if the budget cap bound, or if pressure lasted: then the manager would read the inflated productivity as the new normal. The Try it explores that.

**Try it 9.**

- *Check.* With `use_measured_productivity=1`, what is the perceived productivity at month 30, against the normal 20 patients/clinician/month?
- *Apply.* Make the manager slower to update (perception time 12 months). Does the peak list rise or fall, and why?
- *Extend.* Combine measured productivity with the budget cap of 2,100,000 euro/month. Compare the list at month 120 with planning on normal productivity under the same cap. Does the measurement bias matter now?

<details><summary>Worked answer</summary>

At month 30 the perceived productivity is 20.18 patients/clinician/month, only about 1% above normal: hiring has already relieved most of the pressure. With a perception time of 12 months the peak list falls very slightly, to about 33,910 patients, because the slower belief has not yet caught up with the inflated output when hiring starts. With the budget cap, both rules end at 210 clinicians and the list is almost the same at month 120 (about 38,700 patients): the cap, not the estimate, decides. What the measured rule changes is the manager's story. It states a need of about 238 clinicians, because it divides desired treatments by an inflated productivity; the normal-productivity rule states about 248. Both exceed the 210 the budget pays for, but the measured rule understates the shortfall by about a quarter (28 against 38 clinicians).
</details>

In [ ]:
d = case.run(referral_increase=400, use_measured_productivity=1)
print(f"perceived productivity at month 30: "
      f"{d['Perceived productivity'][30]:.2f}")
d = case.run(referral_increase=400, use_measured_productivity=1,
             perception_time=12)
print(f"perception time 12: peak list {d['Waiting list'].max():,.0f}")
for use in (0, 1):
    d = case.run(referral_increase=400, staff_budget=2100000,
                 use_measured_productivity=use)
    need = d["Desired treatments"] / d["Planning productivity"]
    print(f"cap, measured={use}: list {d['Waiting list'].iloc[-1]:,.0f}, "
          f"clinicians {d['Clinicians'].iloc[-1]:.1f}, stated need "
          f"{need.iloc[-1]:.1f}")

## Check it

Three habits catch most mistakes in decision rules. Run them on every rule you write.

1. **Units.** Every term in a sum must have the same units. In `Desired treatments`, `Expected_referrals` is patients/month and `(Waiting_list - Target_list) / Time_to_reduce_list` is patients over months: the same. `df.attrs["units"]` lists the units of every column.
2. **Extreme conditions.** Set inputs to zero and to ten times normal. With no referrals the list should empty without a clip; a huge referral rate must not make productivity or hiring unbounded.
3. **Halve `dt`.** If the results move, the time step is too long for the fastest process in the model, here the half-month minimum treatment time.

The cell runs all three on the full case model.

In [ ]:
d = case.run(referral_increase=400)
print("units:", {k: d.attrs["units"][k] for k in
                 ["Desired treatments", "Expected referrals", "Net hiring"]})
z = case.run(base_referrals=0)
print(f"no referrals: list at month 120 {z['Waiting list'].iloc[-1]:.1f}, "
      f"clipped: {z.attrs['clipped']}")
h = case.run(referral_increase=36000)
print(f"referrals 10x after month 24: max effect "
      f"{h['Effect of pressure on productivity'].max():.2f}, clinicians "
      f"{h['Clinicians'].iloc[-1]:.0f} (budget allows "
      f"{h['Affordable clinicians'].iloc[-1]:.0f}), clipped: "
      f"{h.attrs['clipped']}")
half = case.run(referral_increase=400, dt=0.0625)
diff = (half["Waiting list"].reindex(d.index) - d["Waiting list"]).abs().max()
print(f"halving dt changes the list by at most {diff:.1f} patients")

The units add up. At ten times the referrals the productivity effect stops at its ceiling of 1.2 and staff stop at the 240 clinicians the budget allows, with no clip. Halving `dt` moves the list by at most 7.6 patients out of tens of thousands.

The zero-referral test finds a real flaw. Nothing goes negative, but the list stops at 13,040 patients and stays there for ever. Once the list is below its target, desired treatments are zero, so the rule lets the clinicians go (fewer than 0.01 remain by month 60) and the remaining patients are never seen. A target meant as a ceiling has become a floor. One repair is a second term: treat at least the list divided by a **longest acceptable wait** of 12 months.

$$
\text{Desired treatments} = \max\!\left(\frac{W}{12},\; \hat{R} + \frac{W - W^{*}}{\tau_W}\right)
$$

In [ ]:
repaired = case.with_equation(
    "Desired treatments", "MAX(Waiting_list / 12, Expected_referrals + "
    "(Waiting_list - Target_list) / Time_to_reduce_list)")
z = repaired.run(base_referrals=0)
gone = z.index[z["Waiting list"] < 1][0]
print(f"repaired, no referrals: list below one patient from month "
      f"{gone:.1f}, clipped: {z.attrs['clipped']}")
d2 = repaired.run(referral_increase=400)
print(f"repaired, 10% rise: largest change in the list "
      f"{(d2['Waiting list'] - d['Waiting list']).abs().max():.1f} patients")

The repaired rule empties the list: it is below one patient from month 20.1, with no clip. In the 10% rise it changes nothing, because the new term never binds while referrals are flowing. Later chapters can keep either form; the point is that only an extreme test exposed the difference.

## Running case

This chapter turned the waiting list from a bathtub with a fixed drain into a managed system:

- `Desired treatments` replace expected referrals and correct the list towards a `Target list` (the stock adjustment logic of section 2).
- `Effect of pressure on productivity` is a `TABLE` through (1, 1) that saturates at 1.2 (section 3).
- `Treatments` are protected by flow, so the list cannot be over-treated (section 6), and desired treatments never fall below the list divided by a longest acceptable wait of 12 months (Check it).
- `Clinicians` are a stock, hired towards `Desired clinicians` and capped by the `Staff budget` (section 7).
- An optional `Perceived productivity` lets the staffing rule plan on measured output (section 9).

Chapter 6, Why things swing, adds the delays in hiring and training that make this rule overshoot and oscillate. Chapter 7 adds fatigue and outsourcing. Chapter 10 calibrates the referrals, which the data never show, against the observed list.

## Tell the decision maker

> Clinicians absorb a rise in demand by working harder, by about 10% in this model, so the list grows much more slowly than the extra referrals alone would suggest. That cushion is not capacity. Plan staff on normal productivity, not on last quarter's output per clinician: under pressure that measure is inflated and makes the staff need look smaller than it is (in this model by about 10 clinicians when the budget binds). A hiring rule that only matches new referrals will hold the list where it peaked; to bring it back, the rule must also aim at the backlog. If the staff budget is below what the rule asks for, say so plainly: the list will keep rising and the staff will stay under pressure.

## Going further

- **Estimating a table from data.** A `TABLE` is a hypothesis about a curve. With paired observations of pressure and productivity, fit the points by least squares with `numpy.linalg.lstsq` on a piecewise-linear basis, and keep the curve monotone and through (1, 1). Chapter 10 shows how to calibrate such parameters against the stock alone.
- **Behavioural experiments.** How people actually make these decisions can be tested in the lab with management flight simulators [@sterman1987; @sterman1989]; anchoring rules fitted to such experiments often explain behaviour better than optimal rules.
- **Best practice.** Formulation guidance, including the principle that decision rules use only available information, is collected in @martinezmoyano2013.
- **The app.** Each molecule in this chapter can be explored with sliders in the LoopBricks app, for example [Weighted Split](https://waelrash1.github.io/loopbricks/#/m/weightedSplit).

## Mini-project

Give the hospital group two clinics: a general clinic with 70% of referrals and a specialist clinic with 30%. Use the case model's parts as a template.

1. Build two waiting lists, each with protected treatments, and one pool of clinicians shared by a **Weighted Split** on the two lists.
2. Give each clinic its own pressure effect with a `TABLE` you can justify; the specialist clinic should saturate earlier (say at 1.1).
3. Add the staffing rule of section 7 on the total desired treatments.
4. Run a 10% referral rise in the specialist clinic only. Compare weights of 1 and 2 for the specialist list. Report the peak and final lists, the clinicians and the productivity in each clinic, and write three sentences for the decision maker.

## Further reading

- Tversky and Kahneman [-@tversky1974] is the original account of anchoring and adjustment as a judgement heuristic.
- Morecroft [-@morecroft1983] explains why system dynamics models decision rules as bounded, partial and based on local information.
- Sterman [-@sterman1989] shows managers using anchoring rules in a simple stock management task, and the oscillations that follow.
- Hines, *Molecules of Structure* [@hines_molecules], is the source of every molecule in this chapter.
- @sterman2000 covers formulating decision rules and nonlinear functions at greater length.